# Building a Retrieval-Augmented Generation (RAG) Pipeline

Welcome to this hands-on workshop! In this script, you'll learn how to build 
a complete RAG pipeline using Hugging Face libraries and open datasets.

In this workshop, we'll create a RAG system that can answer questions about 
a specific domain by:
1. Loading and preparing a knowledge base (dataset)
2. Creating searchable embeddings of our documents
3. Building a vector database for fast retrieval
4. Using a language model to generate answers based on retrieved context

In [1]:
import torch
import faiss
import numpy as np
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

/software/python-anaconda-2022.05-el8-x86_64/envs/transformers/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Environment Setup

In [2]:
# On Midway3, activate the prepared environment with:
# module load python/anaconda-2022.05; source activate transformers

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: NVIDIA H100 NVL
Memory Available: 99.88 GB


# 2. Data Loading and Exploration

For this workshop, we'll use the SQuAD (Stanford Question Answering Dataset) 
as our knowledge base. SQuAD contains Wikipedia articles with associated 
questions and answers, making it perfect for demonstrating RAG.

In [3]:
# Load the SQuAD dataset from local path
dataset_path = "/project/rcc/shared/ai-workshops/squad"
dataset = load_from_disk(dataset_path)

# Use first 1000 examples from train split for the workshop
dataset = dataset['train'].select(range(1000))

# Extract unique contexts (documents) from the dataset
contexts = list(set([example['context'] for example in dataset]))
print(f"\nTotal unique documents: {len(contexts)}")

# Display a sample document
print("\nSample Document:")
print(contexts[0][:500] + "...")  # Show first 500 characters


Total unique documents: 119

Sample Document:
The success of its football team made Notre Dame a household name. The success of Note Dame reflected rising status of Irish Americans and Catholics in the 1920s. Catholics rallied up around the team and listen to the games on the radio, especially when it knocked off the schools that symbolized the Protestant establishment in America — Harvard, Yale, Princeton, and Army. Yet this role as high-profile flagship institution of Catholicism made it an easy target of anti-Catholicism. The most remark...


# 3. Document Chunking

Large documents need to be split into smaller chunks for effective retrieval.
This helps because:
- Embeddings work better on focused, coherent text segments
- Retrieval is more precise when matching specific passages
- We avoid exceeding model token limits

We use a smaller chunk size (100 words) to keep retrieved context concise
and focused, making it easier to understand what the model is using.

In [4]:
def chunk_text(text, chunk_size=100, overlap=20):
    """
    Split text into chunks of approximately chunk_size words with overlap.
    
    Args:
        text: The input text to chunk
        chunk_size: Target number of words per chunk
        overlap: Number of words to overlap between chunks
    
    Returns:
        List of text chunks
    """
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
        
        # Stop if we've reached the end
        if i + chunk_size >= len(words):
            break
    
    return chunks

# Chunk all documents
all_chunks = []
for doc in contexts:
    chunks = chunk_text(doc, chunk_size=100, overlap=20)
    all_chunks.extend(chunks)

print(f"Total documents: {len(contexts)}")
print(f"Total chunks created: {len(all_chunks)}")
print(f"Average chunks per document: {len(all_chunks) / len(contexts):.2f}")

# Show first chunk
print("\nFirst Chunk Example:")
print(all_chunks[0])

Total documents: 119
Total chunks created: 237
Average chunks per document: 1.99

First Chunk Example:
The success of its football team made Notre Dame a household name. The success of Note Dame reflected rising status of Irish Americans and Catholics in the 1920s. Catholics rallied up around the team and listen to the games on the radio, especially when it knocked off the schools that symbolized the Protestant establishment in America — Harvard, Yale, Princeton, and Army. Yet this role as high-profile flagship institution of Catholicism made it an easy target of anti-Catholicism. The most remarkable episode of violence was the clash between Notre Dame students and the Ku Klux Klan in 1924. Nativism and


# 3a. Chunk Quality Checks

Short sanity checks help us spot chunking mistakes early. We'll examine the
distribution of chunk lengths to make sure the overlap behaves as expected.

In [5]:
chunk_lengths = np.array([len(chunk.split()) for chunk in all_chunks])
print("\nChunk length statistics (in words):")
print(f"- Min: {chunk_lengths.min()}")
print(f"- Max: {chunk_lengths.max()}")
print(f"- Mean: {chunk_lengths.mean():.1f}")
print(f"- Std Dev: {chunk_lengths.std():.1f}")

if chunk_lengths.min() == 0:
    print("Warning: Detected empty chunks. Consider adjusting chunk_size/overlap parameters.")
else:
    print("No empty chunks detected.")


Chunk length statistics (in words):
- Min: 21
- Max: 100
- Mean: 80.6
- Std Dev: 26.1
No empty chunks detected.


# 4. Embedding Generation

Embeddings are dense vector representations of text that capture semantic 
meaning. Similar texts will have similar embeddings, enabling us to find 
relevant documents through vector similarity.

## What are embeddings?
- Embeddings convert text into fixed-length numerical vectors (e.g., 1024 dimensions)
- Each dimension captures some aspect of meaning
- Similar concepts are positioned close together in this high-dimensional space
- This allows mathematical operations (like cosine similarity) to measure relevance

## Why do we need embeddings?
- Computers cannot directly compare text semantically
- Embeddings enable efficient similarity search across millions of documents
- They capture context and meaning beyond simple keyword matching

We'll use the BGE (BAAI General Embedding) model, which is state-of-the-art 
for retrieval tasks. The model is pre-loaded locally for this workshop.

In [6]:
# Load the pre-trained embedding model from local path
embedding_model_path = "/project/rcc/shared/ai-workshops/bge-large-en-v1.5"

print("\nLoading BGE embedding model...")
embedding_model = SentenceTransformer(embedding_model_path, device=device)

print(f"Model loaded on: {device}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Generate embeddings for all chunks
print("\nGenerating embeddings for all chunks...")
batch_size = 32

embeddings = embedding_model.encode(
    all_chunks,
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # Normalize for cosine similarity
)

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"First embedding vector length: {len(embeddings[0])}")

# Display sample embeddings to illustrate their structure
print("\nSample text chunk:")
print(all_chunks[0][:200] + "...")

print("\nFirst 5 embeddings (first 20 dimensions each) for comparison:")
for i in range(min(5, len(embeddings))):
    print(f"\nEmbedding {i}: {embeddings[i][:20]}")

print("\nEmbedding statistics for first embedding:")
print(f"- Dimension: {embeddings.shape[1]}")
print(f"- Mean value: {embeddings[0].mean():.6f}")
print(f"- Standard deviation: {embeddings[0].std():.6f}")
print(f"- Vector norm: {np.linalg.norm(embeddings[0]):.6f}")

print("\nComparing embeddings for similar vs. dissimilar chunks:")
# Compare first two chunks (likely similar context)
similarity_close = np.dot(embeddings[0], embeddings[1])
print(f"Similarity between chunk 0 and 1 (likely related): {similarity_close:.4f}")

# Compare distant chunks (likely different topics)
similarity_far = np.dot(embeddings[0], embeddings[-1])
print(f"Similarity between chunk 0 and last chunk (likely unrelated): {similarity_far:.4f}")


Loading BGE embedding model...
Model loaded on: cuda
Embedding dimension: 1024

Generating embeddings for all chunks...


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:05<00:00,  1.55it/s]


Embedding matrix shape: (237, 1024)
First embedding vector length: 1024

Sample text chunk:
The success of its football team made Notre Dame a household name. The success of Note Dame reflected rising status of Irish Americans and Catholics in the 1920s. Catholics rallied up around the team ...

First 5 embeddings (first 20 dimensions each) for comparison:

Embedding 0: [-0.01249329  0.00316251  0.04169437  0.03037924 -0.03513267  0.03841734
 -0.02400298  0.0266265  -0.00084702 -0.01525306  0.00383135  0.02161573
  0.01242508 -0.01009433 -0.03233426 -0.00256163 -0.03559833 -0.01820318
 -0.01282444  0.0182657 ]

Embedding 1: [-0.01580655  0.01934407  0.0233319   0.0446917  -0.02698011  0.02391232
 -0.00018275  0.02161936  0.00360975  0.017581   -0.01360012 -0.00401374
 -0.01645871 -0.02925239 -0.02338663  0.01850159 -0.05554414 -0.01484641
 -0.01916897  0.05254181]

Embedding 2: [-0.0120287  -0.02132192  0.017424    0.02310487 -0.01693153  0.02786976
  0.01138527  0.01859295 -0.0119762

# 5. Vector Database Creation and Persistence

FAISS (Facebook AI Similarity Search) is a library for efficient similarity 
search in large collections of vectors. It creates an index structure that 
allows fast nearest-neighbor queries.

IndexFlatIP computes inner product between vectors. When vectors are 
normalized (unit length), the inner product equals cosine similarity:
  cosine(A, B) = dot(A, B) / (||A|| * ||B||)
  If ||A|| = ||B|| = 1, then cosine(A, B) = dot(A, B)

This is why we normalized embeddings during generation.

In [7]:
# Create FAISS index
embedding_dim = embeddings.shape[1]

# IndexFlatIP computes inner product (cosine similarity with normalized vectors)
index = faiss.IndexFlatIP(embedding_dim)

# Add all embeddings to the index
index.add(embeddings.astype('float32'))

print(f"Number of vectors in index: {index.ntotal}")
print(f"Index dimension: {embedding_dim}")

# Saving the vector database (commented out for workshop)
# In production, you would save the index and chunks to avoid re-embedding

"""
# Save the FAISS index
# faiss.write_index(index, 'squad_embeddings.index')

# Save the text chunks (using pickle or JSON)
# import pickle
# with open('squad_chunks.pkl', 'wb') as f:
#     pickle.dump(all_chunks, f)

# Later, to load:
# index = faiss.read_index('squad_embeddings.index')
# with open('squad_chunks.pkl', 'rb') as f:
#     all_chunks = pickle.load(f)
"""

Number of vectors in index: 237
Index dimension: 1024


"\n# Save the FAISS index\n# faiss.write_index(index, 'squad_embeddings.index')\n\n# Save the text chunks (using pickle or JSON)\n# import pickle\n# with open('squad_chunks.pkl', 'wb') as f:\n#     pickle.dump(all_chunks, f)\n\n# Later, to load:\n# index = faiss.read_index('squad_embeddings.index')\n# with open('squad_chunks.pkl', 'rb') as f:\n#     all_chunks = pickle.load(f)\n"

# 6. Retrieval: Finding Relevant Context

The retrieval process works as follows:
1. Convert the user's query into an embedding (same model as documents)
2. Normalize the query embedding to unit length
3. Compute cosine similarity between query and all document embeddings
4. Return the top-k most similar documents

Normalization is critical:
- Without normalization, longer vectors would dominate similarity scores
- Normalized vectors have unit length (||v|| = 1)
- Inner product of normalized vectors equals cosine similarity
- Cosine similarity ranges from -1 (opposite) to 1 (identical)

In [8]:
def retrieve_context(query, top_k=3):
    """
    Retrieve the most relevant document chunks for a given query.
    
    Args:
        query: The user's question
        top_k: Number of chunks to retrieve
    
    Returns:
        List of retrieved text chunks and their similarity scores
    """
    # For BGE models, add instruction prefix for queries
    query_with_instruction = f"Represent this sentence for searching relevant passages: {query}"
    
    # Encode the query (already normalized by the model)
    query_embedding = embedding_model.encode(
        [query_with_instruction],
        convert_to_numpy=True,
        normalize_embeddings=True  # Ensures unit length for cosine similarity
    )
    
    # Search the index (returns inner product = cosine similarity for normalized vectors)
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    
    # Retrieve the corresponding text chunks
    retrieved_chunks = [all_chunks[idx] for idx in indices[0]]
    
    return retrieved_chunks, distances[0]

# Example query - use a query about Notre Dame since that's in our dataset

example_query = "Where is the University of Notre Dame located?"

print(f"\nQuery: {example_query}\n")
print("Retrieved Context:")

retrieved_docs, scores = retrieve_context(example_query, top_k=3)

for i, (doc, score) in enumerate(zip(retrieved_docs, scores), 1):
    print(f"\n[Chunk {i}] (Similarity: {score:.4f})")
    print(doc)


Query: Where is the University of Notre Dame located?

Retrieved Context:

[Chunk 1] (Similarity: 0.7686)
The University of Notre Dame du Lac (or simply Notre Dame /ˌnoʊtərˈdeɪm/ NOH-tər-DAYM) is a Catholic research university located adjacent to South Bend, Indiana, in the United States. In French, Notre Dame du Lac means "Our Lady of the Lake" and refers to the university's patron saint, the Virgin Mary. The main campus covers 1,250 acres in a suburban setting and it contains a number of recognizable landmarks, such as the Golden Dome, the "Word of Life" mural (commonly known as Touchdown Jesus), and the Basilica.

[Chunk 2] (Similarity: 0.6408)
In 2014 the Notre Dame student body consisted of 12,179 students, with 8,448 undergraduates, 2,138 graduate and professional and 1,593 professional (Law, M.Div., Business, M.Ed.) students. Around 21–24% of students are children of alumni, and although 37% of students come from the Midwestern United States, the student body represents all 50 

# 7. Generative Model Setup

Now we need a language model to generate answers based on the retrieved 
context. We'll use Mistral-Small-24B, a powerful open-source model that's 
been instruction-tuned for following directions and answering questions.

The model is pre-loaded locally for this workshop. Note: This is a large 
model, so loading may take a minute.

In [9]:
# Load the generative model
generator_model_path = "/project/rcc/shared/ai-workshops/Mistral-Small-24B-Instruct-2501"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(generator_model_path)
model = AutoModelForCausalLM.from_pretrained(
    generator_model_path,
    torch_dtype=torch.float16,  # Use half precision for efficiency
    device_map="auto",  # Automatically distribute across available GPUs
    low_cpu_mem_usage=True
)

print("Model loaded successfully!")
print(f"Model device: {model.device}")

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [01:27<00:00,  8.77s/it]

Model loaded successfully!
Model device: cuda:0


# 8. Constructing the Prompt and Generating Answers

To get the best results from our language model, we need to:
1. Format the prompt properly with clear instructions
2. Include retrieved context so the model can reference it
3. Ask the specific question

We'll use a structured prompt template that tells the model to answer based 
only on the provided context.

In [10]:
def construct_prompt(query, context_chunks):
    """
    Construct a prompt for the language model with retrieved context.
    
    Args:
        query: The user's question
        context_chunks: List of retrieved text chunks
    
    Returns:
        Formatted prompt string
    """
    # Combine all context chunks
    context = "\n\n".join([f"[Document {i+1}]\n{chunk}" for i, chunk in enumerate(context_chunks)])
    
    # Create the prompt following Mistral's instruction format
    prompt = f"""<s>[INST] You are a helpful assistant. Answer the question based on the context provided below. If the answer cannot be found in the context, say "I cannot answer this based on the provided context."

Context:
{context}

Question: {query}

Please provide a clear and concise answer. [/INST]"""
    
    return prompt

def generate_answer(query, context_chunks, max_length=200):
    """
    Generate an answer using the language model.
    
    Args:
        query: The user's question
        context_chunks: Retrieved context
        max_length: Maximum length of generated answer
    
    Returns:
        Generated answer string
    """
    # Construct the prompt
    prompt = construct_prompt(query, context_chunks)
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the generated tokens (excluding input prompt)
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    return answer

# Generate answer for our example
answer = generate_answer(example_query, retrieved_docs)

print("\nFinal Answer:")
print(answer)


Final Answer:
The University of Notre Dame is located adjacent to South Bend, Indiana, in the United States.


# 9. Complete RAG Pipeline Function

Let's wrap everything into a single function that performs the entire RAG 
pipeline: retrieval + generation.

In [11]:
def rag_pipeline(query, top_k=3, max_length=200, show_context=True):
    """
    Complete RAG pipeline: retrieve context and generate answer.
    
    Args:
        query: User's question
        top_k: Number of chunks to retrieve
        max_length: Maximum length of generated answer
        show_context: Whether to display retrieved context
    
    Returns:
        Generated answer
    """
    print(f"\nQuestion: {query}\n")
    
    # Step 1: Retrieve relevant context
    context_chunks, scores = retrieve_context(query, top_k=top_k)
    
    if show_context:
        print("\nRetrieved Context:")
        for i, (chunk, score) in enumerate(zip(context_chunks, scores), 1):
            print(f"\n[Chunk {i}] (Similarity: {score:.4f})")
            print(chunk)
    
    # Step 2: Generate answer
    answer = generate_answer(query, context_chunks, max_length=max_length)
    
    print(f"\nAnswer: {answer}\n")

    return answer

# 10. Interactive Experimentation

Test the complete pipeline with various queries related to the knowledge 
base (Wikipedia articles from SQuAD dataset).

In [12]:
# Test the complete pipeline with a Notre Dame-specific query

test_query = "What is the Golden Dome at Notre Dame?"
rag_pipeline(test_query)


Question: What is the Golden Dome at Notre Dame?


Retrieved Context:

[Chunk 1] (Similarity: 0.6760)
The University of Notre Dame du Lac (or simply Notre Dame /ˌnoʊtərˈdeɪm/ NOH-tər-DAYM) is a Catholic research university located adjacent to South Bend, Indiana, in the United States. In French, Notre Dame du Lac means "Our Lady of the Lake" and refers to the university's patron saint, the Virgin Mary. The main campus covers 1,250 acres in a suburban setting and it contains a number of recognizable landmarks, such as the Golden Dome, the "Word of Life" mural (commonly known as Touchdown Jesus), and the Basilica.

[Chunk 2] (Similarity: 0.6083)
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately b

'The Golden Dome at Notre Dame is the gold dome atop the Main Building, which features a golden statue of the Virgin Mary.'

# 11. Multiple Query Examples

In [13]:
# Use questions that are answerable from the Notre Dame context in SQuAD
questions = [
    "When was the College of Science established at Notre Dame?",
    "What does Notre Dame du Lac mean in English?",
    "Who was the legendary football coach at Notre Dame?",
]

for i, q in enumerate(questions, 1):
    print(f"\nQuery {i}/{len(questions)}")
    rag_pipeline(q, top_k=2, max_length=150, show_context=False)
    print()


Query 1/3

Question: When was the College of Science established at Notre Dame?


Answer: The College of Science was established at Notre Dame in 1865.



Query 2/3

Question: What does Notre Dame du Lac mean in English?


Answer: Notre Dame du Lac means "Our Lady of the Lake."



Query 3/3

Question: Who was the legendary football coach at Notre Dame?


Answer: Knute Rockne




# 12. Parameter Tuning: Impact of top_k

The number of chunks retrieved (top_k) significantly affects answer quality.
Too few chunks may miss relevant information, while too many may introduce
noise or exceed context limits. Let's compare different values.

In [14]:
print("\n\nParameter Tuning: Comparing top_k Values")

tuning_query = "What is the main campus landmark at Notre Dame?"

print(f"\nQuery: {tuning_query}\n")

# Compare top_k = 2 vs top_k = 5
for k in [2, 5]:
    print(f"\n--- Using top_k = {k} ---")
    context_chunks, scores = retrieve_context(tuning_query, top_k=k)
    
    print(f"\nRetrieved {k} chunks with similarity scores: {scores}")
    
    answer = generate_answer(tuning_query, context_chunks, max_length=100)
    print(f"\nAnswer: {answer}")
    print()

print("\nObservation:")
print("- With top_k=2: More focused context, potentially faster")
print("- With top_k=5: More comprehensive context, may capture nuances")
print("- Trade-off between precision and recall")



Parameter Tuning: Comparing top_k Values

Query: What is the main campus landmark at Notre Dame?


--- Using top_k = 2 ---

Retrieved 2 chunks with similarity scores: [0.6994026 0.6688191]

Answer: The main campus landmarks at Notre Dame include the Golden Dome, the "Word of Life" mural (commonly known as Touchdown Jesus), and the Basilica.


--- Using top_k = 5 ---

Retrieved 5 chunks with similarity scores: [0.6994026 0.6688191 0.6635151 0.655391  0.6519265]

Answer: The main campus landmarks at Notre Dame include the Golden Dome, the "Word of Life" mural (commonly known as Touchdown Jesus), and the Basilica.


Observation:
- With top_k=2: More focused context, potentially faster
- With top_k=5: More comprehensive context, may capture nuances
- Trade-off between precision and recall


# 13. Parameter Tuning: Impact of Temperature

Temperature controls randomness in generation:
- Low temperature (0.1-0.3): More deterministic, focused answers
- Medium temperature (0.7): Balanced creativity and consistency
- High temperature (1.0+): More diverse but potentially less coherent

In [15]:
print("\n\nParameter Tuning: Comparing Temperature Values")

temp_query = "Who founded Notre Dame?"
context_chunks, _ = retrieve_context(temp_query, top_k=3)

print(f"\nQuery: {temp_query}\n")

for temp in [0.3, 1.0, 1.5]:
    print(f"\n--- Temperature = {temp} ---")
    
    # Generate with specific temperature
    prompt = construct_prompt(temp_query, context_chunks)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=temp,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    print(f"Answer: {answer}")

print("\nObservation:")
print("- Lower temperature: More consistent, factual answers")
print("- Higher temperature: More varied phrasing, occasionally more creative")



Parameter Tuning: Comparing Temperature Values

Query: Who founded Notre Dame?


--- Temperature = 0.3 ---
Answer: Father Edward Sorin of the Congregation of the Holy Cross founded Notre Dame.

--- Temperature = 1.0 ---
Answer: I cannot answer this based on the provided context.

--- Temperature = 1.5 ---
Answer: Based on the provided context, Father Edward Sorin of the Congregation of the Holy Cross founded Notre Dame.

Observation:
- Lower temperature: More consistent, factual answers
- Higher temperature: More varied phrasing, occasionally more creative


# 14. Answer Citation and Source Tracking

In production RAG systems, it's crucial to track which sources were used
to generate each answer. This builds trust and allows verification.

In [16]:
print("\n\nAnswer Citation: Tracking Sources")

def generate_answer_with_citations(query, context_chunks, max_length=200):
    """
    Generate an answer and track which chunks contributed to it.
    
    Returns:
        Tuple of (answer, context_chunks_used)
    """
    # Construct the prompt
    prompt = construct_prompt(query, context_chunks)
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the generated tokens
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    return answer, context_chunks

citation_query = "What does Notre Dame du Lac mean?"

print(f"\nQuery: {citation_query}\n")

# Retrieve context
context_chunks, scores = retrieve_context(citation_query, top_k=3)

# Generate answer with citation tracking
answer, sources = generate_answer_with_citations(citation_query, context_chunks, max_length=100)

print(f"Answer: {answer}\n")
print("Sources used:")
for i, (source, score) in enumerate(zip(sources, scores), 1):
    print(f"\n[Source {i}] (Relevance: {score:.4f})")
    print(source)

print("\nObservation:")
print("- Citations allow users to verify information")
print("- Similarity scores indicate confidence in source relevance")
print("- Critical for transparency in production systems")



Answer Citation: Tracking Sources

Query: What does Notre Dame du Lac mean?

Answer: Notre Dame du Lac means "Our Lady of the Lake."

Sources used:

[Source 1] (Relevance: 0.7720)
The University of Notre Dame du Lac (or simply Notre Dame /ˌnoʊtərˈdeɪm/ NOH-tər-DAYM) is a Catholic research university located adjacent to South Bend, Indiana, in the United States. In French, Notre Dame du Lac means "Our Lady of the Lake" and refers to the university's patron saint, the Virgin Mary. The main campus covers 1,250 acres in a suburban setting and it contains a number of recognizable landmarks, such as the Golden Dome, the "Word of Life" mural (commonly known as Touchdown Jesus), and the Basilica.

[Source 2] (Relevance: 0.5636)
Grotto. The university through the Moreau Seminary has ties to theologian Frederick Buechner. While not Catholic, Buechner has praised writers from Notre Dame and Moreau Seminary created a Buechner Prize for Preaching.

[Source 3] (Relevance: 0.5427)
prominent of whic

# 15. Comparing RAG vs Non-RAG Responses

To understand RAG's value, let's compare answers with and without retrieval.
This demonstrates how retrieval grounds the model in factual information.

In [17]:
print("\n\nComparing RAG vs Non-RAG Responses")

comparison_query = "What was the Notre Dame student enrollment in 2014?"

print(f"\nQuery: {comparison_query}\n")

# Non-RAG: Direct generation without retrieval
print("--- WITHOUT RAG (Model knowledge only) ---")
non_rag_prompt = f"""<s>[INST] You are a helpful assistant. Answer the following question based on your knowledge:

Question: {comparison_query}

Please provide a clear and concise answer. [/INST]"""

inputs = tokenizer(non_rag_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
non_rag_answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"Answer: {non_rag_answer}\n")

# With RAG - show retrieved context to demonstrate grounding
print("--- WITH RAG (Retrieved context + Model) ---")
print(f"\nQuestion: {comparison_query}\n")

# Retrieve and display context
context_chunks, scores = retrieve_context(comparison_query, top_k=3)
print("Retrieved Context:")
for i, (chunk, score) in enumerate(zip(context_chunks, scores), 1):
    print(f"\n[Chunk {i}] (Similarity: {score:.4f})")
    print(chunk)

# Generate answer
answer = generate_answer(comparison_query, context_chunks, max_length=150)
print(f"\nAnswer: {answer}\n")

print("Observation:")
print("- Non-RAG may provide general or outdated information")
print("- RAG grounds answers in specific, current documents")
print("- RAG provides verifiable facts from the knowledge base")
print("- Retrieved context shows exactly where the information comes from")



Comparing RAG vs Non-RAG Responses

Query: What was the Notre Dame student enrollment in 2014?

--- WITHOUT RAG (Model knowledge only) ---
Answer: As of my last update in October 2023, I don't have real-time data or access to specific enrollment numbers for Notre Dame for 2014. However, according to general information available up to that point, Notre Dame University typically has an undergraduate student population of around 8,000 to 8,500 students. For precise figures, you would need to consult Notre Dame's official records or archives.

--- WITH RAG (Retrieved context + Model) ---

Question: What was the Notre Dame student enrollment in 2014?

Retrieved Context:

[Chunk 1] (Similarity: 0.7750)
In 2014 the Notre Dame student body consisted of 12,179 students, with 8,448 undergraduates, 2,138 graduate and professional and 1,593 professional (Law, M.Div., Business, M.Ed.) students. Around 21–24% of students are children of alumni, and although 37% of students come from the Midwester

# 16. Putting It All Together: Key RAG Concepts

This final section demonstrates the core concepts that make RAG effective
by examining different aspects of the pipeline we've built.

In [18]:
print("\n\nPutting It All Together: Key RAG Concepts")

# Concept 1: Semantic vs Keyword Search
print("\n1. Semantic Search vs Keyword Matching")
print("\nSemantic search understands meaning, not just exact words.")

semantic_query = "What is Notre Dame's main architectural feature?"
print(f"\nQuery: '{semantic_query}'")
print("(Note: Uses words like 'architectural feature' rather than exact terms)")

context_chunks, scores = retrieve_context(semantic_query, top_k=2)
print(f"\nTop retrieved chunk (score: {scores[0]:.4f}):")
print(context_chunks[0])

answer = generate_answer(semantic_query, context_chunks, max_length=100)
print(f"\nAnswer: {answer}")

# Concept 2: Handling Ambiguity
print("\n\n2. Handling Ambiguous Queries")
print("\nRAG retrieves relevant context to disambiguate unclear questions.")

ambiguous_query = "What happened in 1842?"
print(f"\nQuery: '{ambiguous_query}'")
print("(This could refer to many events)")

context_chunks, scores = retrieve_context(ambiguous_query, top_k=2)
print(f"\nRetrieved context provides specific information:")
for i, (chunk, score) in enumerate(zip(context_chunks, scores), 1):
    print(f"\n[Chunk {i}] (Similarity: {score:.4f})")
    print(chunk)

answer = generate_answer(ambiguous_query, context_chunks, max_length=100)
print(f"\nAnswer: {answer}")

# Concept 3: Retrieval Quality Matters
print("\n\n3. Why Retrieval Quality Matters")
print("\nLet's examine how similarity scores reflect relevance.")

quality_query = "What are Notre Dame's academic divisions?"
print(f"\nQuery: '{quality_query}'")

context_chunks, scores = retrieve_context(quality_query, top_k=5)
print(f"\nSimilarity scores for top 5 chunks:")
for i, score in enumerate(scores, 1):
    print(f"Chunk {i}: {score:.4f}")

print(f"\nScore range: {scores.max():.4f} to {scores.min():.4f}")
print(f"Difference between best and worst: {scores.max() - scores.min():.4f}")

if scores[0] > scores[-1] + 0.1:
    print("\nStrong top match - high confidence in retrieval")
else:
    print("\nSimilar scores - multiple relevant chunks found")

print("\nBest matching chunk:")
print(context_chunks[0])



Putting It All Together: Key RAG Concepts

1. Semantic Search vs Keyword Matching

Semantic search understands meaning, not just exact words.

Query: 'What is Notre Dame's main architectural feature?'
(Note: Uses words like 'architectural feature' rather than exact terms)

Top retrieved chunk (score: 0.6582):
The School of Architecture was established in 1899, although degrees in architecture were first awarded by the university in 1898. Today the school, housed in Bond Hall, offers a five-year undergraduate program leading to the Bachelor of Architecture degree. All undergraduate students study the third year of the program in Rome. The university is globally recognized for its Notre Dame School of Architecture, a faculty that teaches (pre-modernist) traditional and classical architecture and urban planning (e.g. following the principles of New Urbanism and New Classical Architecture). It also awards the renowned annual Driehaus Architecture Prize.

Answer: I cannot answer this base

# Summary

This workshop demonstrated a complete RAG pipeline that:

1. Loads and processes a knowledge base
2. Creates semantic embeddings for efficient search
3. Implements fast similarity search with FAISS
4. Retrieves relevant context for queries
5. Generates answers using a large language model

## Key Technical Points:
- Embeddings convert text into vectors that capture semantic meaning
- Normalization enables cosine similarity via inner product
- Vector databases enable efficient search across large document collections
- Proper prompt engineering improves answer quality
- RAG grounds language models in factual, retrieved information

## Next Steps:
- Experiment with different chunk sizes and overlap
- Try different embedding models for your domain
- Tune top_k and temperature for your use case
- Implement hybrid search (semantic + keyword)
- Add re-ranking for improved retrieval quality